In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torchxrayvision albumentations

Mounted at /content/drive


In [ ]:
"""B3 and B4 for ML4H 2026 Paper 331 camera-ready (Nathan).

Run in Colab with a GPU, after mounting Drive and installing torchxrayvision
and albumentations (see the run steps). Uses the exact model, preprocessing,
masks and scorer of the paper pipeline (Full_Grad_CAM notebook and the B1
rebuttal cell).

B3  Re-scores the existing epoch-25 7x7 Grad-CAM maps (gradcam_final_npy_25,
    exported by the B1 cell with LAYER="final") with nearest-neighbour instead
    of bilinear upsampling. No model needed. Gate: bilinear re-score must match
    the paper's per-seed means in T1_per_seed.csv within 0.0001 on every seed.

B4  One forward/backward per image on the 21 best-validation checkpoints gives
    both the 7x7 Grad-CAM map (paper method) and the vanilla gradient map
    (|d logit_pneumonia / d input|, the Captum Saliency default). Gates:
      1. Regenerated epoch-25 vanilla gradient maps must match the existing
         grad_vanilla_npy_25 files for the first GATE_IMAGES patients of
         GATE_SEEDS seeds.
      2. Existing epoch-25 vanilla gradient maps must score mean 0.369, SD 0.008.
      3. Best-arm Grad-CAM must score mean 0.328, SD 0.033 (tracker B4).

Outputs go to OUT_DIR: B3_B4_per_seed.csv, B3_B4_per_image.csv,
B3_B4_summary.json, B3_B4_report.txt.
"""

import json
import os
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import albumentations as A
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchxrayvision as xrv
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score

ROOT = "/content/drive/MyDrive/AI_Research/Model_Results/SaliencyRun/xrv_pipeline_nb"
LABELS = "/content/drive/MyDrive/AI_Research/stage_2_train_labels.csv"
ZIP_PATH = "/content/drive/MyDrive/AI_Research/rsna-pneumonia-dataset-npy.zip"
EXTRACT_DIR = "/content/rsna-pneumonia-dataset-npy"
NPY_MAP = "/content/drive/MyDrive/AI_Research/npy_map.json"
GC_FINAL_NPY = f"{ROOT}/gradcam_final_npy_25"
VG_FINAL_NPY = f"{ROOT}/grad_vanilla_npy_25"
GC_T1 = f"{ROOT}/paper_outputs/T1_per_seed.csv"
GC_BEST_NPY = f"{ROOT}/gradcam_final_npy_best"
VG_BEST_NPY = f"{ROOT}/grad_vanilla_npy_best"
OUT_DIR = f"{ROOT}/paper_outputs/B3_B4"

SCORE = 1024
IMG_SIZE = 224
PNEUMONIA_LOGIT = 1
XRV_WEIGHTS = "densenet121-res224-mimic_nb"
DROPOUT = 0.3366734543720239
CHANCE_AUPRC = 0.117
N_BOOT = 1000
BOOT_SEED = 0
IMAGE_SD_THRESHOLD = 0.02
N_SEEDS = 21
N_IMAGES = 601
GATE_SEEDS = 2
LOAD_THREADS = 64  # changed: parallel Drive reads
GATE_IMAGES = 20
SKLEARN_CHECK_IMAGES = 5
REFERENCE = {
    "vg_final": {"mean": 0.369, "sd": 0.008},
    "gc_best": {"mean": 0.328, "sd": 0.033},
}
AGG_TOL = 0.0006
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

val_transform_xrv = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])
npy_map = {}


class XRVClassifier(nn.Module):
    """Identical to the training notebook."""

    def __init__(self, xrv_weights, dropout, num_classes=2):
        super().__init__()
        backbone = xrv.models.DenseNet(weights=xrv_weights)
        self.features = backbone.features
        num_features = backbone.classifier.in_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(num_features, num_classes))

    def forward(self, x):
        feats = torch.relu(self.features(x))
        return self.classifier(feats.mean(dim=(2, 3)))


def build_model(path):
    model = XRVClassifier(XRV_WEIGHTS, DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
    return model.eval()


def ckpt_path(seed, tag):
    return f"{ROOT}/frozen_xrv_seed{seed}_{tag}.pth"


def load_input(pid):
    """Identical to RSNADatasetXRV with val_transform_xrv."""
    img = np.load(npy_map[pid]).astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = (img * 255.0).clip(0, 255).astype(np.uint8)
    img = val_transform_xrv(image=img)["image"].astype(np.float32)
    img = xrv.datasets.normalize(img, 255)
    return torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()


def cam_and_saliency(model, x):
    """One backward pass: paper 7x7 Grad-CAM (raw, channel mean) and |input gradient|."""
    model.zero_grad(set_to_none=True)
    x = x.clone().requires_grad_(True)
    acts = torch.relu(model.features(x))
    acts.retain_grad()
    logits = model.classifier(acts.mean(dim=(2, 3)))
    logits[0, PNEUMONIA_LOGIT].backward()
    with torch.no_grad():
        w = acts.grad.mean(dim=(0, 2, 3)).view(1, -1, 1, 1)
        cam = (acts * w).mean(dim=1)[0]
        sal = x.grad.abs()[0, 0]
    return cam.cpu().numpy().astype(np.float32), sal.cpu().numpy().astype(np.float32)


def build_masks(patient_ids):
    """Identical to m1_scoring_cell."""
    lab = pd.read_csv(LABELS)
    lab = lab[lab["Target"] == 1].dropna(subset=["x", "y", "width", "height"])
    groups = {pid: g for pid, g in lab.groupby("patientId")}
    masks = {}
    for pid in patient_ids:
        m = np.zeros((SCORE, SCORE), dtype=bool)
        for _, r in groups[pid].iterrows():
            x0, y0 = int(round(r.x)), int(round(r.y))
            x1, y1 = int(round(r.x + r.width)), int(round(r.y + r.height))
            m[y0:y1, x0:x1] = True
        assert m.any(), pid
        masks[pid] = m
    return masks


def prep_map(raw, mode):
    """Scorer preprocessing: rectify, divide by max, upsample to 1024."""
    a = np.maximum(np.asarray(raw, dtype=np.float32).squeeze(), 0.0)
    a = a / max(a.max(), 1e-8)
    t = torch.from_numpy(a)[None, None].to(DEVICE)
    if mode == "bilinear":
        up = F.interpolate(t, size=(SCORE, SCORE), mode="bilinear", align_corners=False)
    else:
        up = F.interpolate(t, size=(SCORE, SCORE), mode="nearest")
    return up[0, 0]


def fast_ap(scores, labels):
    """average_precision_score on the GPU, same tie handling as sklearn."""
    s = scores.reshape(-1).double()
    order = torch.argsort(s, descending=True)
    s = s[order]
    y = labels.reshape(-1)[order].double()
    last = torch.nonzero(s[1:] != s[:-1]).flatten()
    last = torch.cat([last, torch.tensor([s.numel() - 1], device=s.device)])
    tps = torch.cumsum(y, 0)[last]
    fps = (last + 1).double() - tps
    recall = tps / tps[-1]
    precision = tps / (tps + fps)
    prev = torch.cat([torch.zeros(1, dtype=torch.float64, device=s.device), recall[:-1]])
    return float(torch.sum((recall - prev) * precision))


def score(raw, mask_t, mode, check=False):
    up = prep_map(raw, mode)
    ap = fast_ap(up, mask_t)
    if check:
        ref = average_precision_score(mask_t.cpu().numpy().ravel(), up.cpu().numpy().ravel())
        assert abs(ap - ref) < 1e-9, f"GPU scorer differs from sklearn: {ap} vs {ref}"
    py, px = np.unravel_index(int(torch.argmax(up)), (SCORE, SCORE))
    return ap, bool(mask_t[py, px])


def load_many(paths):
    """Read many small .npy files in parallel (Drive reads are latency-bound)."""
    with ThreadPoolExecutor(LOAD_THREADS) as ex:
        return list(ex.map(np.load, paths))


def score_folder(name, seeds, patient_ids, mask_ts, path_fn, mode, rows):
    """Score every (seed, patient) map from disk; append per-image rows; return per-seed means.

    Each finished condition is cached to OUT_DIR/cache_<name>.csv and reused on rerun.
    """
    cache = f"{OUT_DIR}/cache_{name}.csv"
    if os.path.exists(cache):
        df = pd.read_csv(cache)
        if df["seed"].nunique() == len(seeds) and len(df) == len(seeds) * len(patient_ids):
            rows.extend(df.to_dict("records"))
            print(f"  [{name}] loaded from cache")
            return df.groupby("seed")["auprc"].mean().loc[seeds]
    means, new_rows = {}, []
    for i, s in enumerate(seeds, 1):
        t_load = time.time()
        raws = load_many([path_fn(s, pid) for pid in patient_ids])
        t_score = time.time()
        aps, hits = [], []
        for j, (pid, raw) in enumerate(zip(patient_ids, raws)):
            ap, hit = score(raw, mask_ts[pid], mode, check=(i == 1 and j < SKLEARN_CHECK_IMAGES))
            aps.append(ap)
            hits.append(hit)
            new_rows.append({"condition": name, "seed": s, "patientId": pid, "auprc": ap, "pointing_hit": hit})
        means[s] = float(np.mean(aps))
        print(f"  [{name}] {i:2d}/{len(seeds)} seed {s:<5d} AUPRC {means[s]:.4f}  pointing {np.mean(hits):.3f}  "
              f"(load {t_score - t_load:.0f}s, score {time.time() - t_score:.0f}s)")
    pd.DataFrame(new_rows).to_csv(cache, index=False)
    rows.extend(new_rows)
    return pd.Series(means)


def boot_idx(n):
    rng = np.random.default_rng(BOOT_SEED)
    return np.stack([rng.choice(n, n, replace=True) for _ in range(N_BOOT)])


def ci(v):
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]


def summarize(series, rows, name):
    v = series.to_numpy()
    idx = boot_idx(len(v))
    mean, sd = float(v.mean()), float(np.std(v, ddof=1))
    per_img = pd.DataFrame([r for r in rows if r["condition"] == name]).groupby("patientId")["auprc"].std(ddof=1)
    hits = pd.DataFrame([r for r in rows if r["condition"] == name]).groupby("seed")["pointing_hit"].mean()
    return {
        "mean": mean,
        "seed_sd": sd,
        "seed_sd_ci95": ci(np.std(v[idx], axis=1, ddof=1)),
        "sd_over_headroom": sd / (mean - CHANCE_AUPRC),
        "per_seed_min": float(v.min()),
        "per_seed_max": float(v.max()),
        "quantiles_5_25_50_75_95": [float(q) for q in np.quantile(v, [0.05, 0.25, 0.5, 0.75, 0.95])],
        "frac_images_sd_gt_0.02": float((per_img > IMAGE_SD_THRESHOLD).mean()),
        "per_image_sd_median": float(per_img.median()),
        "pointing_range": [float(hits.min()), float(hits.max())],
    }


def paired(a, b):
    """a minus b, paired by seed, same bootstrap draws for both."""
    seeds = sorted(a.index)
    x, y = a.loc[seeds].to_numpy(), b.loc[seeds].to_numpy()
    idx = boot_idx(len(seeds))
    rho, p = spearmanr(x, y)
    return {
        "n_seeds": len(seeds),
        "mean_diff": float(x.mean() - y.mean()),
        "mean_diff_ci95": ci(x[idx].mean(axis=1) - y[idx].mean(axis=1)),
        "sd_diff": float(np.std(x, ddof=1) - np.std(y, ddof=1)),
        "sd_diff_ci95": ci(np.std(x[idx], axis=1, ddof=1) - np.std(y[idx], axis=1, ddof=1)),
        "seeds_a_greater": int((x > y).sum()),
        "max_abs_seed_diff": float(np.abs(x - y).max()),
        "spearman_rho": float(rho),
        "spearman_p": float(p),
    }


def agg_gate(summary, key):
    ref = REFERENCE[key]
    ok = abs(summary["mean"] - ref["mean"]) <= AGG_TOL and abs(summary["seed_sd"] - ref["sd"]) <= AGG_TOL
    return {"expected": ref, "got": {"mean": round(summary["mean"], 4), "sd": round(summary["seed_sd"], 4)}, "pass": bool(ok)}


def export_best(seeds, patient_ids, inputs):
    """Grad-CAM and vanilla gradient maps for the best-validation checkpoints, resumable."""
    for i, s in enumerate(seeds, 1):
        dg, dv = f"{GC_BEST_NPY}/seed_{s}", f"{VG_BEST_NPY}/seed_{s}"
        os.makedirs(dg, exist_ok=True)
        os.makedirs(dv, exist_ok=True)
        todo = [p for p in patient_ids if not (os.path.exists(f"{dg}/{p}.npy") and os.path.exists(f"{dv}/{p}.npy"))]
        if not todo:
            print(f"  [export best] {i:2d}/{len(seeds)} seed {s} already exported")
            continue
        model = build_model(ckpt_path(s, "best"))
        for pid in todo:
            cam, sal = cam_and_saliency(model, inputs[pid])
            np.save(f"{dg}/{pid}.npy", cam)
            np.save(f"{dv}/{pid}.npy", sal)
        del model
        torch.cuda.empty_cache()
        print(f"  [export best] {i:2d}/{len(seeds)} seed {s} exported {len(todo)}")


def vg_file_gate(seeds, patient_ids, inputs):
    """Regenerate a few epoch-25 vanilla gradient maps and compare with the files used in the paper."""
    worst, shapes = 0.0, set()
    for s in seeds[:GATE_SEEDS]:
        model = build_model(ckpt_path(s, "epoch25"))
        for pid in patient_ids[:GATE_IMAGES]:
            _, sal = cam_and_saliency(model, inputs[pid])
            ref = np.load(f"{VG_FINAL_NPY}/seed_{s}/{pid}.npy").astype(np.float32).squeeze()
            shapes.add(tuple(ref.shape))
            if ref.shape != sal.shape:
                return {"pass": False, "reason": f"shape {ref.shape} on disk vs {sal.shape} regenerated"}
            rel = float(np.abs(sal - ref).max() / max(np.abs(ref).max(), 1e-12))
            worst = max(worst, rel)
        del model
        torch.cuda.empty_cache()
    return {"pass": worst < 1e-3, "max_relative_diff": worst, "disk_shape": [list(x) for x in shapes]}


def write_report(out):
    lines = []

    def f(x):
        return f"{x:.4f}"

    def c(v):
        return f"[{f(v[0])}, {f(v[1])}]"

    def block(title, s):
        lines.append(f"  {title}: mean {f(s['mean'])}, seed SD {f(s['seed_sd'])} (95% CI {c(s['seed_sd_ci95'])}), "
                     f"SD/headroom {s['sd_over_headroom']:.2f}, range {f(s['per_seed_min'])} to {f(s['per_seed_max'])}, "
                     f"quantiles 5/25/50/75/95 = {', '.join(f(q) for q in s['quantiles_5_25_50_75_95'])}, "
                     f"images with seed SD > 0.02: {s['frac_images_sd_gt_0.02']:.0%}")

    def pair(title, d):
        lines.append(f"  {title}: mean diff {f(d['mean_diff'])} (95% CI {c(d['mean_diff_ci95'])}), "
                     f"SD diff {f(d['sd_diff'])} (95% CI {c(d['sd_diff_ci95'])}), first higher on {d['seeds_a_greater']}/{d['n_seeds']} seeds, "
                     f"seed-rank Spearman {d['spearman_rho']:.2f} (p = {d['spearman_p']:.3g})")

    g = out["gates"]
    lines.append("GATES")
    for k, v in g.items():
        lines.append(f"  {k}: {'PASS' if v['pass'] else 'FAIL'}  {json.dumps({a: b for a, b in v.items() if a != 'pass'})}")
    lines.append("")
    lines.append("B3  epoch-25 Grad-CAM 7x7, bilinear vs nearest-neighbour upsampling (AUPRC)")
    block("bilinear", out["B3"]["bilinear"])
    block("nearest ", out["B3"]["nearest"])
    pair("nearest minus bilinear", out["B3"]["nearest_minus_bilinear"])
    lines.append("")
    lines.append("B4  best-validation arm (AUPRC)")
    block("vanilla gradient, best", out["B4"]["vg_best"])
    block("Grad-CAM, best        ", out["B4"]["gc_best"])
    block("vanilla gradient, ep25", out["B4"]["vg_final"])
    pair("best arm: VG minus Grad-CAM", out["B4"]["best_vg_minus_gc"])
    pair("epoch-25 arm: VG minus Grad-CAM", out["B4"]["final_vg_minus_gc"])
    pair("VG: best minus epoch-25", out["B4"]["vg_best_minus_final"])
    pair("Grad-CAM: best minus epoch-25", out["B4"]["gc_best_minus_final"])
    text = "\n".join(lines)
    with open(f"{OUT_DIR}/B3_B4_report.txt", "w") as fh:
        fh.write(text + "\n")
    print("\n" + "=" * 80 + "\n" + text + "\n" + "=" * 80)


def main():
    global npy_map
    t0 = time.time()
    assert os.path.exists("/content/drive/MyDrive"), "Mount Drive in the notebook first: drive.mount('/content/drive')"
    os.makedirs(OUT_DIR, exist_ok=True)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    seeds = sorted(int(d.split("_")[1]) for d in os.listdir(VG_FINAL_NPY) if d.startswith("seed_"))
    assert len(seeds) == N_SEEDS, f"found {len(seeds)} seeds"
    patient_ids = sorted(f[:-4] for f in os.listdir(f"{VG_FINAL_NPY}/seed_{seeds[0]}") if f.endswith(".npy"))
    assert len(patient_ids) == N_IMAGES, f"found {len(patient_ids)} patients"
    for s in seeds:
        for tag in ("best", "epoch25"):
            assert os.path.exists(ckpt_path(s, tag)), f"missing {ckpt_path(s, tag)}"
        assert os.path.isdir(f"{GC_FINAL_NPY}/seed_{s}"), f"missing {GC_FINAL_NPY}/seed_{s}; run the B1 cell with LAYER='final'"
    print(f"{len(seeds)} seeds, {len(patient_ids)} patients, device {DEVICE}")

    if not os.path.exists(EXTRACT_DIR):
        print("extracting dataset zip...")
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall("/content")
    with open(NPY_MAP) as fh:
        npy_map = json.load(fh)
    inputs = {pid: load_input(pid).to(DEVICE) for pid in patient_ids}
    masks = build_masks(patient_ids)
    mask_ts = {pid: torch.from_numpy(m).to(DEVICE) for pid, m in masks.items()}
    print(f"mean box fraction {np.mean([m.mean() for m in masks.values()]):.4f} (chance floor in paper {CHANCE_AUPRC})")

    gates = {}
    print("\nGate: regenerate epoch-25 vanilla gradient maps and compare with grad_vanilla_npy_25")
    gates["vg_regeneration_matches_disk"] = vg_file_gate(seeds, patient_ids, inputs)
    print("  ", gates["vg_regeneration_matches_disk"])
    if not gates["vg_regeneration_matches_disk"]["pass"]:
        print("  WARNING: vanilla gradient method differs from the paper's export. B4 vanilla gradient numbers are not valid.")

    print("\nExport best-validation maps")
    export_best(seeds, patient_ids, inputs)

    rows = []
    print("\nScoring")
    per_seed = {
        "gc_final_bilinear": score_folder("gc_final_bilinear", seeds, patient_ids, mask_ts,
                                          lambda s, p: f"{GC_FINAL_NPY}/seed_{s}/{p}.npy", "bilinear", rows),
        "gc_final_nearest": score_folder("gc_final_nearest", seeds, patient_ids, mask_ts,
                                         lambda s, p: f"{GC_FINAL_NPY}/seed_{s}/{p}.npy", "nearest", rows),
        "vg_final": score_folder("vg_final", seeds, patient_ids, mask_ts,
                                 lambda s, p: f"{VG_FINAL_NPY}/seed_{s}/{p}.npy", "bilinear", rows),
        "gc_best": score_folder("gc_best", seeds, patient_ids, mask_ts,
                                lambda s, p: f"{GC_BEST_NPY}/seed_{s}/{p}.npy", "bilinear", rows),
        "vg_best": score_folder("vg_best", seeds, patient_ids, mask_ts,
                                lambda s, p: f"{VG_BEST_NPY}/seed_{s}/{p}.npy", "bilinear", rows),
    }

    t1 = pd.read_csv(GC_T1).set_index("seed")["auprc_mean"].loc[seeds]
    worst = float((per_seed["gc_final_bilinear"] - t1.values).abs().max())
    gates["gc_final_bilinear_matches_T1_per_seed"] = {"max_abs_diff": worst, "pass": worst < 1e-4}

    S = {k: summarize(v, rows, k) for k, v in per_seed.items()}
    gates["vg_final_matches_paper"] = agg_gate(S["vg_final"], "vg_final")
    gates["gc_best_matches_tracker"] = agg_gate(S["gc_best"], "gc_best")

    out = {
        "gates": gates,
        "B3": {"bilinear": S["gc_final_bilinear"], "nearest": S["gc_final_nearest"],
               "nearest_minus_bilinear": paired(per_seed["gc_final_nearest"], per_seed["gc_final_bilinear"])},
        "B4": {"vg_best": S["vg_best"], "gc_best": S["gc_best"], "vg_final": S["vg_final"],
               "best_vg_minus_gc": paired(per_seed["vg_best"], per_seed["gc_best"]),
               "final_vg_minus_gc": paired(per_seed["vg_final"], per_seed["gc_final_bilinear"]),
               "vg_best_minus_final": paired(per_seed["vg_best"], per_seed["vg_final"]),
               "gc_best_minus_final": paired(per_seed["gc_best"], per_seed["gc_final_bilinear"])},
    }
    pd.DataFrame(per_seed).rename_axis("seed").to_csv(f"{OUT_DIR}/B3_B4_per_seed.csv")
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/B3_B4_per_image.csv", index=False)
    with open(f"{OUT_DIR}/B3_B4_summary.json", "w") as fh:
        json.dump(out, fh, indent=2)
    write_report(out)
    print(f"\nwrote {OUT_DIR}  ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()

21 seeds, 601 patients, device cuda
extracting dataset zip...
mean box fraction 0.1174 (chance floor in paper 0.117)

Gate: regenerate epoch-25 vanilla gradient maps and compare with grad_vanilla_npy_25
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/mimic_nb-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/mimic_nb-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
   {'pass': False, 'max_relative_diff': 0.20753885805606842, 'disk_shape': [[224, 224]]}

Export best-validation maps
  [export best]  1/21 seed 7 already exported
  [export best]  2/21 seed 13 already exported
  [export best]  3/21 seed 21 already exported
  [export best]  4/21 seed 42 already exported
  [export best]  5/21 seed 99 already exported
  [export best]  6/21 seed 123 already exported
  [export best]  7/21 seed 256 already exported
  [export best]  8/21 seed 314 al

In [ ]:
import numpy as np, torch, torch.nn.functional as F, os
from sklearn.metrics import average_precision_score

seeds_d = sorted(int(d.split("_")[1]) for d in os.listdir(VG_FINAL_NPY) if d.startswith("seed_"))
pids_d = sorted(f[:-4] for f in os.listdir(f"{VG_FINAL_NPY}/seed_{seeds_d[0]}") if f.endswith(".npy"))[:10]
masks_d = build_masks(pids_d)

def grad_map(model, x, target, amp=False, times_input=False):
    model.zero_grad(set_to_none=True)
    x = x.clone().requires_grad_(True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        logits = model(x)
    logits = logits.float()
    if target == "logit1":
        out = logits[0, 1]
    elif target == "logit_diff":
        out = logits[0, 1] - logits[0, 0]
    elif target == "prob1":
        out = torch.softmax(logits, 1)[0, 1]
    else:
        out = torch.log_softmax(logits, 1)[0, 1]
    out.backward()
    g = x.grad[0, 0]
    if times_input:
        g = g * x.detach()[0, 0]
    return g.abs().detach().cpu().numpy().astype(np.float32)

def norm(a):
    a = np.maximum(np.asarray(a, dtype=np.float32).squeeze(), 0)
    return a / max(a.max(), 1e-8)

def ap(a, m):
    up = F.interpolate(torch.from_numpy(norm(a))[None, None], size=(SCORE, SCORE), mode="bilinear", align_corners=False)[0, 0].numpy()
    return average_precision_score(m.ravel(), up.ravel())

candidates = {
    "logit1 (current)": dict(target="logit1", tf32=True),
    "logit1, TF32 off": dict(target="logit1", tf32=False),
    "logit1, fp16 autocast": dict(target="logit1", tf32=True, amp=True),
    "logit1 - logit0": dict(target="logit_diff", tf32=False),
    "softmax prob1": dict(target="prob1", tf32=False),
    "log-softmax prob1": dict(target="logprob1", tf32=False),
    "grad x input": dict(target="logit1", tf32=False, times_input=True),
}

s = seeds_d[0]
model = build_model(ckpt_path(s, "epoch25"))
disk = {p: np.load(f"{VG_FINAL_NPY}/seed_{s}/{p}.npy") for p in pids_d}
disk_ap = {p: ap(disk[p], masks_d[p]) for p in pids_d}
print(f"seed {s}, {len(pids_d)} images, disk dtype {disk[pids_d[0]].dtype}, disk max {np.abs(disk[pids_d[0]]).max():.4g}\n")
print(f"{'method':24s} {'pearson r':>10s} {'max diff':>10s} {'mean |dAUPRC|':>14s} {'scale disk/new':>15s}")
for name, c in candidates.items():
    torch.backends.cuda.matmul.allow_tf32 = c["tf32"]
    torch.backends.cudnn.allow_tf32 = c["tf32"]
    rs, mds, daps, scales = [], [], [], []
    for p in pids_d:
        new = grad_map(model, load_input(p).to(DEVICE), c["target"], c.get("amp", False), c.get("times_input", False))
        a, b = norm(disk[p]), norm(new)
        rs.append(np.corrcoef(a.ravel(), b.ravel())[0, 1])
        mds.append(np.abs(a - b).max())
        daps.append(abs(ap(new, masks_d[p]) - disk_ap[p]))
        scales.append(np.abs(disk[p]).max() / max(new.max(), 1e-12))
    print(f"{name:24s} {np.mean(rs):10.6f} {np.max(mds):10.4f} {np.mean(daps):14.6f} {np.median(scales):15.4g}")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

seed 7, 10 images, disk dtype float32, disk max 0.0006411

method                    pearson r   max diff  mean |dAUPRC|  scale disk/new
logit1 (current)           0.998213     0.1526       0.000823           1.004
logit1, TF32 off           0.998236     0.1521       0.000901           1.009
logit1, fp16 autocast      0.998504     0.2048       0.000873           1.004
logit1 - logit0            0.997728     0.1533       0.001208          0.5201
softmax prob1              0.997728     0.1533       0.001207           12.76
log-softmax prob1          0.997728     0.1533       0.001208           5.967
grad x input               0.811963     0.9663       0.069631        0.002545
